# Task 12-2

Преобразование фотографии в стиль Monet с помощью предобученной модели CycleGAN (`style_monet`) и вычисление среднего значения интенсивности пикселей в палитре RGB при помощи OpenCV.

Итог: в конце ячейка печатает одно число, округленное до тысячных.

In [4]:
from pathlib import Path
import os
import subprocess
import sys
import urllib.request

# Ставим зависимости, если они отсутствуют в окружении ноутбука.
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "torch",
        "numpy",
        "torchvision",
        "opencv-python",
        "dominate",
        "visdom",
        "wandb",
    ],
    check=True,
)

import cv2


BASE_DIR = Path.cwd()
if BASE_DIR.name != "task-12-2" and (BASE_DIR / "task-12-2").exists():
    BASE_DIR = BASE_DIR / "task-12-2"

image_path = BASE_DIR / "image.jpg"
if not image_path.exists():
    image_path = BASE_DIR / "image.png"
if not image_path.exists():
    raise FileNotFoundError("В папке task-12-2 не найдено image.jpg или image.png")

repo_dir = BASE_DIR / "pytorch-CycleGAN-and-pix2pix"
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/junyanz/pytorch-CycleGAN-and-pix2pix", str(repo_dir)],
        check=True,
    )

checkpoints_dir = repo_dir / "checkpoints" / "style_monet_pretrained"
checkpoints_dir.mkdir(parents=True, exist_ok=True)
model_path = checkpoints_dir / "latest_net_G.pth"
model_url = "http://efrosgans.eecs.berkeley.edu/cyclegan/pretrained_models/style_monet.pth"
if not model_path.exists():
    print("Скачиваю style_monet pretrained model...")
    urllib.request.urlretrieve(model_url, model_path)

testC_dir = repo_dir / "datasets" / "monet2photo" / "testC"
os.makedirs(testC_dir, exist_ok=True)

for f in testC_dir.iterdir():
    if f.is_file():
        f.unlink()

# Приводим вход к имени и формату из референсного примера.
src_bgr = cv2.imread(str(image_path))
if src_bgr is None:
    raise RuntimeError(f"OpenCV не смог прочитать исходное изображение: {image_path}")
dst_image = testC_dir / "ITMO_1.png"
cv2.imwrite(str(dst_image), src_bgr)

def run_and_get_means(direction: str):
    cmd = [
        sys.executable,
        "test.py",
        "--dataroot",
        str(testC_dir),
        "--name",
        "style_monet_pretrained",
        "--model",
        "test",
        "--no_dropout",
        "--direction",
        direction,
    ]
    res = subprocess.run(cmd, cwd=repo_dir, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(f"CycleGAN test.py ({direction}) завершился с ошибкой:\n{res.stderr}")

    fake_path = (
        repo_dir
        / "results"
        / "style_monet_pretrained"
        / "test_latest"
        / "images"
        / "ITMO_1_fake.png"
    )
    real_path = (
        repo_dir
        / "results"
        / "style_monet_pretrained"
        / "test_latest"
        / "images"
        / "ITMO_1_real.png"
    )

    if not fake_path.exists() or not real_path.exists():
        raise FileNotFoundError("Не найдены файлы ITMO_1_fake.png / ITMO_1_real.png")

    fake_bgr = cv2.imread(str(fake_path))
    real_bgr = cv2.imread(str(real_path))
    if fake_bgr is None or real_bgr is None:
        raise RuntimeError("OpenCV не смог прочитать результат(ы) преобразования")

    fake_rgb = cv2.cvtColor(fake_bgr, cv2.COLOR_BGR2RGB)
    real_rgb = cv2.cvtColor(real_bgr, cv2.COLOR_BGR2RGB)
    return round(float(fake_rgb.mean()), 3), round(float(real_rgb.mean()), 3)

mean_atob_fake, mean_atob_real = run_and_get_means("AtoB")
mean_btoa_fake, mean_btoa_real = run_and_get_means("BtoA")

print(f"AtoB fake mean: {mean_atob_fake}")
print(f"BtoA fake mean: {mean_btoa_fake}")
print(f"AtoB real mean: {mean_atob_real}")
print(f"BtoA real mean: {mean_btoa_real}")

# Обычно для задачи про 'фото -> Monet' используется AtoB fake.
print(f"\nCandidate answer (AtoB fake): {mean_atob_fake}")


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


AtoB fake mean: 144.702
BtoA fake mean: 144.702
AtoB real mean: 122.787
BtoA real mean: 122.787

Candidate answer (AtoB fake): 144.702
